---
title: "数据分析思路沉淀"
author: "KaisMemo"
date: "2026-02-20"
categories: [data analysis]
---

这篇博文记录了我对于数据分析的一些方法论或者思路的沉淀。

## 数据分析的经典工作流

1. 业务理解与问题定义
2. 数据获取
3. 数据描述与探索
4. 数据清洗
5. 特征工程
6. 数据建模与分析
7. 评估与验证
8. 结论、策略与落地

这个流程是一个迭代循环的过程。在实际操作中，可能会在“数据清洗”阶段发现需要更多的“数据获取”，或者在“建模”后发现需要重新回到“业务理解”调整目标。

## 第一步：业务理解与问题定位

这是最开始也是最关键的一步。目标是明确要解决什么问题。需要界定相关的指标有哪些，并统一各个指标的统计口径。不同业务场景的有特定的指标体系。

### 电商

核心逻辑是：流量 $\rightarrow$ 转化 $\rightarrow$ 留存 $\rightarrow$ 利润。常见的指标或体系模型如下：

- GMV (Gross Merchandise Volume，交易总额)：平台成交总额（含未支付/退款）。
- CR (Conversion Rate，转化率)：转化用户数 / 访问用户数。
- CAC (Customer Acquisition Cost，获客成本)：总营销费用 / 新客人数。
- AOV (Average Order Value，客单价)：总销售额 / 总订单数。
- 动销率：有销量的商品种数 / 经营的总商品种数。
- RFM 模型：最近一次消费 (Recency)、频率 (Frequency)、金额 (Monetary) 的用户分层

### 内容/社交平台

核心逻辑是：吸引关注 $\rightarrow$ 提升粘性 $\rightarrow$ 商业变现。常见的指标或体系模型如下：

- DAU / MAU (Daily Active User / Monthly Active User)：日活与月活的比值（反映产品“粘性”）
- 互动率：(点赞+评论+分享) / 阅读量。
- 人均时长：用户在 App 内的总时长 / 活跃人数。
- CTR (Click-Through Rate，点击率)：内容曝光后的点击比例。
- 次留 / 七留：用户在第 N 天后再次访问的比例。

### SaaS/订阅服务

核心逻辑是：订阅增长 $\rightarrow$ 减少流失 $\rightarrow$ 复利效应。

- MRR / ARR (Monthly Recurring Revenue / Annual Recurring Revenue)：月度/年度经常性收入（核心钱袋子）。
- LTV (Lifetime Value，客户终身价值)：用户生命周期内预计带来的净利润。
- Churn Rate（流失率）：本月流失客户数 / 期初总客户数。
- NDR (Net Dollar Retention，净收入留存)：老客户产生的收入增长（含升档订阅）。
- LTV / CAC（单位经济效益）；大于 3 说明商业模式健康，大于 5 说明投入不足。

### 通用用户增长模型：AARRR

1. 获客 (Acquisition): 渠道转化率、CPA (Cost Per Acquisition，单人获客成本)。
2. 激活 (Activation): 关键功能使用率（如：社交产品加 5 个好友）。
3. 留存 (Retention): 周留存、月留存。
4. 变现 (Revenue): ARPU (单用户平均收入)、付费率。
5. 传播 (Referral): K-Factor (病毒系数，即每个老用户带来的新用户数)。

同一个指标不同的人对它的解释方式不一样，为了统一口径，比较成熟的组织中，会有数据字典来定义每个指标的计算方式。

## 第二步：数据获取

根据定义的问题，从不同渠道提取原始数据。使用 SQL 从 OLTP、OLAP、Hive/Spark 中获取数据；或者使用爬虫、第三方 API 等。

SQL 各个子句的执行顺序：

1. `FROM / JOIN`，确定数据来源，进行表连接，生成原始数据集。
2. `WHERE`，对原始数据进行初步过滤，不能使用聚合函数。
3. `GROUP BY`，将过滤后的数据按照指定的列进行分组。
4. `WITH ROLLUP / CUBE`，（可选）生成分组后的汇总行。
5. `HAVING`，对分组后的结果进行过滤，可以使用聚合函数。
6. `SELECT`，确定要显示的列，计算表达式，并处理别名。
7. `DISTINCT`，对选择出的结果集进行去重。
8. `ORDER BY`，对最终结果进行排序。
9. `LIMIT / OFFSET`，截取最终展示的行数。

绝大多数主流关系型数据库（MySQL, PostgreSQL, Oracle, SQL Server, SQLite）都支持这些核心语法。但是但各厂商为了竞争、性能优化或历史原因，发展出了各自的“方言”（Dialects）。不要死记硬背方言，需要用到特定数据库的函数时，直接查官方文档（Cheat Sheet）即可。会用标准 SQL 的话，切换数据库时的上手成本通常不到 20%。

你发现某个 SQL 查询逻辑每周、每天都要跑，且口径已经和业务方对齐时，那么把它用视图或者实表固化下来是最佳实践。视图的好处是没有数据冗余，但是每次查询都需要重新计算，实表的好处是一次计算，多次查询，但是会有数据冗余。在数据量比较大或者查询非常频繁（BI 看板）的时候，用实表会更好。

对于爬虫方式来说，最成熟的方式是使用 Scrapy，但是在实际的数据分析项目中，还是优先寻找官方或隐藏的 API 接口比较好。很多看起来需要爬虫的网站，通过浏览器 F12 开发者工具的 Network 面板，往往能找到返回 JSON 数据的后端接口。直接用 Requests 请求这些接口，效率会比解析 HTML 高得多。

这里通常遵循"大聚靠 SQL，小调靠 Python"的原则：在 SQL 中完成（过滤掉异常订单、关联维度表、按用户/天进行初步聚合）。将 SQL 产出的“轻度聚合表”导入 Python 环境，进行后续的建模和策略推断。

## 第三步：数据描述与探索

在深入处理前，先对数据有个直观认识。查看数据行数、列名、数据类型，然后观察均值、中位数、标准差，识别数据的偏态，并通过散点图、直方图观察变量间的初步关系。

对于分类文本数据，可以将之理解为离散标签的数据。在后续的特征工程初期，可以对这些分类列进行编码，转换为数值。

EDA 之后，应该能回答这三个问题：

- 这盘数能用吗？（质量检查）
- 目前业务大概是什么样子？（基准认知）
- 接下来的建模/分析应该重点关注哪些维度？（策略方向）

## 第四步：数据清洗

原始数据通常是“脏”的，需要通过以下步骤变成“干净”的数据：

- 去除重复值
- 删除、填充（均值/众数）或保留缺失值
- 通过盖帽法或业务逻辑剔除异常数据点
- 统一时间格式
- 纠正错别字

填充缺失值的时候，需要分情况处理，如果缺失值很少，可以直接把对应的行删除，如果绝大部分都是缺失值，且缺失值没有明确的业务意义，可以直接把这列删除。否则，可以用均值、众数填充缺失值。也就是说，不是所有缺失值都需要补全，除了缺失数量很少的情况，如果缺失值本身就代表了一些业务含义，那么可以直接填充一个类似“Unknown”之类的。

均值和众数填充可以涵盖绝大部分场景，但是也可以用随机森林或者 KNN 这类机器学习算法来填充这些缺失值，这个虽然比较 Fasion，但是需要注意数据泄露的问题。需要在机器学习填充之前，先把数据集拆分成训练集和测试集，用训练集练出来的 Imputer 填充训练集和测试集合，绝对不能用测试集训练任何模型。

经验法则是，除非你在打算法竞赛，否则不要使用机器学习补全。

## 插曲：常用的机器学习算法及其数据要求

在数据分析与挖掘领域，机器学习算法是实现从“描述现状”到“预测未来”跨越的核心工具。不同的算法有不同的适用场景和数据要求。

其中，监督学习和无监督学习的区别在于是否存在“标准答案”（标签/Label）。

在机器学习语境下，“监督”指的是在训练过程中，模型不仅能看到输入的特征数据（$X$），还能看到这些数据对应的正确输出结果（$Y$）。

可以把这个过程想象成一个学生在做一套带有参考答案的习题集。学生（模型）根据题目（特征）给出预测答案，然后对比参考答案（标签）。如果错了，学生就根据误差（Loss）来调整自己的解题思路（权重参数），直到预测值与标准答案尽可能接近。

无监督学习中不存在“监督者”，数据中没有标签，模型只能“自生自灭”地去寻找规律。核心逻辑是通过分析数据的内在结构、相似性或分布特征，将数据自动划分为不同的群体或简化表达。

### 监督学习：回归

这类算法用于预测连续型的数值（如房价、销售额、气温）。

- 线性回归 (Linear Regression)

  - 大致原理：寻找一个线性方程 $y = w_1x_1 + w_2x_2 + ... + b$，使得预测值与实际值之间的平方误差和（MSE）最小。
  - 数据要求：自变量与因变量间存在线性趋势；残差需符合正态分布、独立性及方差齐性；对异常值较为敏感。
  - 应用场景：电商大促销售额预测、房地产估值分析、广告投入与投资回报率（ROI）建模。

- 岭回归与 Lasso 回归 (Regularized Regression)

  - 大致原理：在线性回归的基础上加入惩罚项（L1 或 L2 正则化），用于处理多重共线性问题并防止模型过拟合。
  - 数据要求：特征维度较高且存在相关性。
  - 应用场景：基因数据分析、具有大量宏观经济指标的预测模型。

### 监督学习：分类

用于将数据点分配到预定义的类别中（如：是/否、买/不买、违约/不违约）。

- 逻辑回归 (Logistic Regression)

  - 大致原理：虽名为回归，实为分类。它通过 Sigmoid 函数将线性组合映射到 $[0, 1]$ 区间，表示某事件发生的概率。
  - 数据要求：要求特征间多重共线性较低；适用于线性可分或近似线性可分的数据。
  - 应用场景：用户流失预测（二分类）、金融风控中的信用评分卡、垃圾邮件检测。

- 支持向量机 (SVM)

  - 大致原理：在特征空间中寻找一个超平面，使得不同类别样本之间的间隔（Margin）最大化。通过核函数（Kernel）可处理非线性分类。
  - 数据要求：数据量不宜过大（训练复杂度高）；对特征缩放（Scaling）非常敏感。
  - 应用场景：文本分类、图像识别、生物信息学中的蛋白质分类。

- K-近邻算法 (KNN)

  - 大致原理：对于新样本，计算其与训练集中最近的 $k$ 个样本的距离，由这 $k$ 个邻居的多数票决定其类别。
  - 数据要求：必须进行标准化处理；样本量极大时查询效率低。
  - 应用场景：简单的手写体识别、基于地理位置的服务推荐。

### 基于树的集成算法

这类算法因其强大的非线性拟合能力和对特征工程要求较低的特性，是目前数据竞赛和工业界最常用的方案。

- 决策树 (Decision Tree)

  - 大致原理：通过信息增益（ID3）、信息增益比（C4.5）或基尼系数（CART）递归地切分数据，形成树状结构。
  - 数据要求：不需要特征缩放；易产生过拟合（Overfitting）。
  - 应用场景：初步探索特征重要性、贷款审批逻辑生成。

- 随机森林 (Random Forest)

  - 大致原理：构建多棵独立的决策树，每棵树使用随机抽取的样本和特征进行训练，最后通过投票或平均得出结果。
  - 数据要求：对异常值和噪声具有极强的鲁棒性；不容易过拟合。
  - 应用场景：用户流失预警、金融风控评分卡、特征重要性评估。

- 梯度提升决策树 (GBDT, XGBoost, LightGBM, CatBoost)

  - 大致原理：每一棵树都在拟合上一棵树的残差（误差），通过不断迭代使损失函数最小化。
  - 数据要求：调参复杂度较高；对非线性关系的建模能力极强。
  - 应用场景：电商精准营销、搜索排名优化、复杂的金融时序预测

### 非监督学习：聚类与降维

用于在无标签数据中发现潜在的结构或模式。

- K-Means 聚类

  - 大致原理：通过迭代寻找 $K$ 个簇中心，使得每个点到其所属簇中心的欧氏距离之和最小。
  - 数据要求：数值型数据；需要预先设定 $K$ 值；对特征缩放（Standardization）非常敏感，因为涉及距离计算。
  - 应用场景：用户画像分群（RFM 模型后的细分）、门店选址分析、异常检测。

- 主成分分析 (PCA)

  - 大致原理：通过正交变换将高维变量转换为少数几个线性不相关的综合指标（主成分），在保留大部分信息的前提下简化数据。
  - 数据要求：变量间应具有相关性；适用于数值型连续变量。
  - 应用场景：多指标评价体系降维、可视化高维数据、预处理步骤（消除特征间相关性）。

### 关联规则分析

虽然有时被归为传统的统计挖掘，但在商业数据分析中地位极高。

- Apriori / FP-Growth 算法

  - 大致原理：挖掘数据集中项与项之间的频繁出现关系，通过支持度（Support）、置信度（Confidence）和提升度（Lift）来衡量规则。
  - 数据要求：事务型数据（如购物篮流水）；对离散化要求高。
  - 应用场景：超市货架摆放优化（“啤酒与尿布”）、电商交叉销售（Cross-selling）建议、网页路径点击流分析。

## 第五步：特征工程

为了让数据更具解释力或提升模型效果。需要从原始数据中创造新指标（如：从出生日期计算年龄）提取新特征，通过归一化、标准化或对数变换等进行特征变换，通过维度缩减保留对目标变量影响最大的特征。

- 归一化 (Normalization)

  归一化通常是指将数值压缩到 $[0, 1]$ 区间。当需要使用对距离敏感的算法时，如 $K$-近邻算法（KNN）、支持向量机（SVM），或者为了加速梯度下降的收敛速度，一般会进行归一化处理。公式很简单：$$x_{new} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

- 标准化 (Standardization)

  标准化将数据转换为均值为 0，标准差为 1 的分布（$Z$-Score）。对于多数机器学习算法，如逻辑回归、线性回归、主成分分析（PCA）。它比归一化更稳健，因为它不会受极端孤立值（Outliers）的剧烈影响。另外，它可以消除量纲差异，例如一个特征是“收入”（万级别），另一个是“年龄”（十位级别），标准化能让它们在同一量级上竞争。公式：$$z = \frac{x - \mu}{\sigma}$$（其中 $\mu$ 为均值，$\sigma$ 为标准差）

- 对数变换 (Log Transformation)

  通过取对数（通常是 $\ln$ 或 $\log_{10}$）来压缩长尾分布的数据。当数据呈现“右偏”分布（大部分值很小，极少数值极大，如收入、点击量、电商订单金额）时考虑做对数变换。另外，当特征与目标值之间呈指数增长关系时，对数变换能将其转为线性关系，方便回归模型处理。而且它也可以消除异方差性，减少数据的波动幅度。公式：$$y = \ln(x + 1)$$

- 维度缩减 (Dimension Reduction)

  减少特征的数量，同时尽可能保留原始数据的有效信息。当特征数量接近甚至超过样本数量，会导致“维度灾难”，此时需要做维度缩减。另外，如果特征之间高度相关，这反映出冗余信息过多，也需要缩减维度。特征选取方式主要有这么几种：

  - 主成分分析 (PCA)：最经典的线性降维方法，通过旋转坐标轴找到方差最大的投影方向。
  - 特征选择 (Feature Selection)：通过相关性系数、互信息（Mutual Information）或随机森林的特征重要性评分，直接剔除不重要的列。
  - 线性判别分析 (LDA)：有监督的降维，旨在最大化类间差异。

## 第六步：数据建模与分析

根据问题类型选择合适的方法。

- 对于描述性分析（如：过去发生了什么？），可以用报表或者看板
- 对于诊断性分析（如：为什么发生？），可以用下钻分析或者相关性分析
- 对于预测性分析（如：未来可能发生什么？），可以用回归、分类或时间序列分析
- 对于处方性分析（如：我们该如何做？），可以用 A/B Testing 或优化算法

不过，当你层层拆解 KPI 遇到瓶颈（比如维度太多，或者变量之间互相关联）时，诊断和预测都可以用机器学习做，比较方便快捷。预测和诊断其实是机器学习这枚硬币的两面。预测是向前看。给模型输入 $X$（特征），让它告诉你 $Y$（结果）未来会是多少；诊断是向后看。当我们已经有了 $Y$（结果），利用模型去拆解到底哪一个 $X$ 对 $Y$ 的贡献最大。


## 第七步：评估与验证

确保分析结果的准确性与稳健性。需要做显著性检验，判断结果是随机产生的还是具有统计学意义，另外，需要判断分析结果是否符合业务常识，如果是机器学习，需检查准确率、召回率等指标。

## 第八步：结论、策略与落地

通过可视化图表和 PPT 展示核心发现，将数据语言转化为业务策略（如：建议针对某人群定向发放优惠券），策略执行后，进行复盘分析，形成闭环。